# Factor campaign 01 - 2026-09-30

Plan 04 phase 6. Six library factors plus the equal-weight composite `score`, universe members from 2019-01,
each winsorised, standardised and neutralised on Shenwan L1 + log market cap (D-093). Reports come from
`make evaluate` (derived/reports/factor_campaign_01/); this notebook only reads them through the package,
applies the FACTOR_TESTING_DESIGN §6 bar (`FactorAcceptance`) and draws them (`FactorChart`).

Forward returns run from the open of t+1 to the close of t+h. Newey-West t is prewhitened (D-090). The
campaign-level Holm / BHY / Romano-Wolf adjustment covers all 7 factors x 5 horizons.

In [1]:
import pandas as pd

from quant_cn.core.config import Config
from quant_cn.core.frames import FACTOR_PANEL, IC_SERIES, IC_TEST, QUANTILE_RETURNS
from quant_cn.factor.factor_acceptance import FactorAcceptance
from quant_cn.lake.lake_catalog import LakeCatalog
from quant_cn.lake.lake_query import LakeQuery
from quant_cn.service_builder import ServiceBuilder
from quant_cn.statistic.block_bootstrap import BlockBootstrap
from quant_cn.statistic.hac_variance import HacVariance
from quant_cn.statistic.ic_calculator import IcCalculator
from quant_cn.statistic.ic_significance import IcSignificance
from quant_cn.visualization.chart_theme import ChartTheme
from quant_cn.visualization.factor_chart import FactorChart

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)
config = Config.load()
campaign = config.campaign
catalog = LakeCatalog(config.lake.root, config.datasets, read_only=True)
query = LakeQuery(catalog)
reports = config.lake.root / "derived" / "reports" / campaign.id
ic_series = IC_SERIES.validate(pd.read_parquet(reports / "ic_series.parquet"))
ic_test = IC_TEST.validate(pd.read_parquet(reports / "ic_test.parquet"))
quantiles = QUANTILE_RETURNS.validate(pd.read_parquet(reports / "quantiles.parquet"))
trials = pd.read_parquet(config.lake.root / "meta" / "trials" / f"{campaign.id}.parquet")
theme = ChartTheme.from_mode("light")
print(f"campaign {campaign.id}: {ic_series['factor'].nunique()} factors, horizons {campaign.horizons}, "
      f"{len(ic_test)} test rows, {len(trials)} trials, IC dates {ic_series['trade_date'].min()} -> "
      f"{ic_series['trade_date'].max()}")

campaign factor_campaign_01: 7 factors, horizons [1, 5, 10, 20, 60], 350 test rows, 35 trials, IC dates 20190102 -> 20260929


## 1. Verdicts at h = 20 (FACTOR_TESTING_DESIGN §6, criteria 1-3)
1. NW t >= 3.0 (HLZ) **and** BHY-adjusted p <= 0.05; 2. same sign with t >= 1.5 in 2019-21, 2022-24 and 2025-;
3. bootstrap 95% CI of ICIR excludes 0. Criterion 4 (DSR, PBO, out-of-sample) needs a traded variant: plan 06.

In [2]:
verdicts = FactorAcceptance(horizon=20).compute(ic_test)
cols = ["factor", "ic_mean", "t", "p_bhy", "t_2019-2021", "t_2022-2024", "t_2025-", "icir", "icir_ci_lo",
        "icir_ci_hi", "verdict", "failed"]
display(verdicts[cols])
for r in verdicts.itertuples():
    reason = "criteria 1-3 met" if r.verdict == "pass" else f"fails criterion {r.failed}"
    print(f"{r.factor:<18} {r.verdict.upper():<5} IC {r.ic_mean:+.3f}  t {r.t:+.2f}  BHY p {r.p_bhy:.3f}  -> {reason}")

,factor,ic_mean,t,p_bhy,t_2019-2021,t_2022-2024,t_2025-,icir,icir_ci_lo,icir_ci_hi,verdict,failed
0,reversal_1m,0.0517,4.8962,0.0000,3.1941,3.0686,2.1194,0.5682,0.4208,0.7188,pass,
1,turnover_60d,0.0786,4.8450,0.0000,3.2611,3.4835,1.6460,0.8104,0.6387,0.9890,pass,
2,score,0.0610,3.5876,0.0031,3.5761,1.4658,2.0086,0.7129,0.5271,0.9248,fail,2
3,ep_ttm,0.0361,1.8645,0.3938,1.5082,1.0238,1.0027,0.4071,0.2336,0.5953,fail,"1,2"
4,roe_ttm,0.0173,0.9626,1.0000,0.9432,0.2080,1.1402,0.2191,0.0407,0.3985,fail,"1,2"
5,profit_growth_ttm,0.0093,0.7726,1.0000,0.8054,0.1246,0.4916,0.1863,0.0078,0.3579,fail,"1,2"
6,momentum_12_1,-0.0047,-0.2411,1.0000,0.0310,-0.0585,-0.4328,-0.0467,-0.2080,0.1170,fail,"1,2,3"


reversal_1m        PASS  IC +0.052  t +4.90  BHY p 0.000  -> criteria 1-3 met
turnover_60d       PASS  IC +0.079  t +4.85  BHY p 0.000  -> criteria 1-3 met
score              FAIL  IC +0.061  t +3.59  BHY p 0.003  -> fails criterion 2
ep_ttm             FAIL  IC +0.036  t +1.86  BHY p 0.394  -> fails criterion 1,2
roe_ttm            FAIL  IC +0.017  t +0.96  BHY p 1.000  -> fails criterion 1,2
profit_growth_ttm  FAIL  IC +0.009  t +0.77  BHY p 1.000  -> fails criterion 1,2
momentum_12_1      FAIL  IC -0.005  t -0.24  BHY p 1.000  -> fails criterion 1,2,3


**Library status** (recorded in FACTOR_LIBRARY §10):
- `turnover_60d`, `reversal_1m` -> **accepted** (criteria 1-3; criterion 4 pending plan 06).
- `score` -> **composite-input**: criterion 2 missed by 0.03 in 2022-24.
- `ep_ttm`, `roe_ttm`, `profit_growth_ttm`, `momentum_12_1` -> **archived at h = 20** with these reports;
  momentum: no 12-1 momentum in A-shares (reversal-dominated).

## 2. ICIR by factor, h = 20

In [3]:
FactorChart(theme, "icir", horizon=20).render(ic_test).show()

## 3. Horizon profile (D-095)
`peak_horizon` = argmax of mean IC over {1, 5, 10, 20, 60}; the half-life is fitted only at or beyond the peak
and is NA ("rising") when IC is still highest at 60 sessions; `accumulation_ratio` = IC_60 / IC_20.
A rising profile is a note for the holding-period choice (plan 06), not a failure.

In [4]:
profile = ic_test[(ic_test["metric"] == "ic_mean") & (ic_test["window"] == "full")].pivot(
    index="factor", columns="horizon", values="value")
profile.columns = [f"ic_h{int(h)}" for h in profile.columns]
notes = verdicts.set_index("factor")[["peak_horizon", "half_life", "accumulation_ratio"]]
profile.join(notes).assign(shape=lambda d: d["half_life"].map(lambda v: "rising" if pd.isna(v) else "decaying"))

,ic_h1,ic_h5,ic_h10,ic_h20,ic_h60,peak_horizon,half_life,accumulation_ratio,shape
factor,,,,,,,,,
ep_ttm,0.0101,0.0242,0.0295,0.0361,0.0497,60.0000,NaN,1.3738,rising
momentum_12_1,0.0004,-0.0018,-0.0028,-0.0047,-0.0091,60.0000,NaN,1.9125,rising
profit_growth_ttm,0.0032,0.0060,0.0071,0.0093,0.0097,60.0000,NaN,1.0446,rising
reversal_1m,0.0151,0.0421,0.0489,0.0517,0.0415,20.0000,126.5687,0.8033,decaying
roe_ttm,0.0034,0.0116,0.0143,0.0173,0.0209,60.0000,NaN,1.2042,rising
score,0.0122,0.0409,0.0509,0.0610,0.0724,60.0000,NaN,1.1872,rising
turnover_60d,0.0084,0.0448,0.0600,0.0786,0.1135,60.0000,NaN,1.4435,rising


In [5]:
for name in verdicts["factor"]:
    FactorChart(theme, "horizon", name, height=360).render(ic_test).show()

## 4. Rolling IC, h = 20 (accepted factors and the composite)
252-session rolling mean with a band of +/- 1.96 sd / sqrt(252 / 20): the rolling window holds about 12.6
non-overlapping 20-session observations.

In [6]:
for name in ["turnover_60d", "reversal_1m", "score"]:
    FactorChart(theme, "ic_series", name, horizon=20).render(ic_series).show()

## 5. Quantile ladders, h = 20
Mean 20-session forward return per quintile (Q5 = highest processed factor value), equal weight, every date
(overlapping); long-short (Q5 - Q1) in the title.

In [7]:
for name in verdicts["factor"]:
    FactorChart(theme, "quantiles", name, horizon=20, height=340).render(quantiles).show()

In [8]:
ls = ic_test[ic_test["metric"].isin(["ls_mean", "ls_sharpe", "rank_autocorr"])]
ls.pivot_table(index="factor", columns=["metric", "horizon"], values="value")

metric            ls_mean                                 ls_sharpe                                 rank_autocorr
horizon           1.0000  5.0000  10.0000 20.0000 60.0000   1.0000  5.0000  10.0000 20.0000 60.0000       1.0000 
factor                                                                                                           
ep_ttm            -0.0004  0.0001  0.0005  0.0011  0.0025   -1.2886  0.0713  0.1293  0.1327  0.0842        0.9981
momentum_12_1      0.0003  0.0009  0.0012  0.0026  0.0067    0.7470  0.3706  0.2215  0.2523  0.2249        0.9917
profit_growth_ttm  0.0001  0.0006  0.0013  0.0026  0.0073    0.6565  0.6088  0.5731  0.5948  0.5285        0.9955
reversal_1m       -0.0006  0.0033  0.0060  0.0107  0.0236   -1.3120  1.3439  1.1399  1.0215  1.0766        0.9281
roe_ttm           -0.0003 -0.0001 -0.0001 -0.0002 -0.0009   -1.0157 -0.0609 -0.0269 -0.0264 -0.0297        0.9985
score             -0.0007  0.0021  0.0044  0.0094  0.0241   -2.1408  1.0927  1.0636  1.0778  0.8558        0.9848
turnover_60d      -0.0011  0.0018  0.0051  0.0111  0.0350   -2.4415  0.7763  1.0229  1.1913  1.8319        0.9993

## 6. Does neutralisation remove value? Raw vs neutralised IC for the fundamentals
Neutralised = the campaign value (industry + size neutral). Raw = the same universe with the unprocessed
`value_raw`; winsorising and standardising are monotone within a date, so the raw rank IC equals the rank IC
before neutralisation. Same IcCalculator / IcSignificance settings (h = 20, forward return from the open of
t+1, prewhitened NW, 2,000 block-bootstrap draws). **Diagnostic only: not recorded in the TrialLog and not a
candidate for acceptance.** If EP is alive within industries, its neutralised IC should hold up against raw.

In [9]:
fundamentals = ["ep_ttm", "roe_ttm", "profit_growth_ttm"]
raw = query.sql(
    "SELECT trade_date, ts_code, factor, value_raw AS value, value_raw FROM derived.features "
    "WHERE factor IN ('ep_ttm', 'roe_ttm', 'profit_growth_ttm') AND trade_date >= ?", [campaign.start])
raw = FACTOR_PANEL.validate(FACTOR_PANEL.normalize(raw, strict=True))
reader = ServiceBuilder().build_reader(config, catalog)
prices = reader.read_prices(campaign.start, "99991231", dense=True)
calc = IcCalculator()
raw_ic = calc.compute(raw, calc.build_forward_returns(prices, [20]))
sig = IcSignificance(HacVariance(), BlockBootstrap(resamples=2000, seed=0))
raw_rows = pd.concat([sig.compute(raw_ic[raw_ic["factor"] == f]) for f in fundamentals])
keep = ["factor", "window", "value", "t"]
pick = lambda d: d[(d["metric"] == "ic_mean") & (d["horizon"] == 20.0) & d["factor"].isin(fundamentals)][keep]
compare = pick(ic_test).merge(pick(raw_rows), on=["factor", "window"], suffixes=("_neutral", "_raw"))
compare = compare.rename(columns={"value_neutral": "ic_neutral", "value_raw": "ic_raw"})
compare["kept"] = compare["ic_neutral"] / compare["ic_raw"]
compare.sort_values(["factor", "window"])

,factor,window,ic_neutral,t_neutral,ic_raw,t_raw,kept
1,ep_ttm,2019-2021,0.0352,1.5082,0.0363,1.2760,0.9702
2,ep_ttm,2022-2024,0.0398,1.0238,0.0273,0.4323,1.4549
3,ep_ttm,2025-,0.0314,1.0027,0.0066,0.1257,4.7862
0,ep_ttm,full,0.0361,1.8645,0.0263,0.9172,1.3728
5,profit_growth_ttm,2019-2021,0.0166,0.8054,0.0169,0.4852,0.9815
6,profit_growth_ttm,2022-2024,0.0023,0.1246,-0.0058,-0.1824,-0.3970
7,profit_growth_ttm,2025-,0.0085,0.4916,-0.0014,-0.0458,-6.1397
4,profit_growth_ttm,full,0.0093,0.7726,0.0041,0.2054,2.2788
9,roe_ttm,2019-2021,0.0256,0.9432,0.0230,0.5276,1.1134
10,roe_ttm,2022-2024,0.0070,0.2080,-0.0143,-0.2617,-0.4859


In [10]:
for r in compare[compare["window"] == "full"].itertuples():
    print(f"{r.factor:<18} raw IC {r.ic_raw:+.3f} (t {r.t_raw:+.2f})  ->  neutralised {r.ic_neutral:+.3f} "
          f"(t {r.t_neutral:+.2f}); neutral / raw = {r.kept:.2f}")

ep_ttm             raw IC +0.026 (t +0.92)  ->  neutralised +0.036 (t +1.86); neutral / raw = 1.37
profit_growth_ttm  raw IC +0.004 (t +0.21)  ->  neutralised +0.009 (t +0.77); neutral / raw = 2.28
roe_ttm            raw IC +0.003 (t +0.09)  ->  neutralised +0.017 (t +0.96); neutral / raw = 6.41
